# Use Case 6: Electronics Sales Processing

## # Name: Chandra Suresh Reddy V
## # Portal ID : 10364839

In [0]:
initial_path = "/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv"

B. Initial Load and Bronze Requirements

•	Read and inspect electronics_sales_initial.csv. Display sample rows, schema, columns and source count.

•	Create the Bronze DataFrame/table and add source_file, batch_id and ingestion_timestamp.

•	Use BATCH_001 for the initial file.

•	Apply the validation rules below and create a rejection_reason for each rejected row.

•	Persist valid Bronze records and rejected records separately.


In [0]:
electronics_df = spark.read.csv(
    "/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv",
    header=True,
    inferSchema=True
)

In [0]:
display(electronics_df)

sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status
SAL0001,2026-10-08,C001,Product 2,SONY,Coimbatore,2,375.0,20.0,2,RETURNED
SAL0002,2026-10-09,C002,Product 3,LG,Bengaluru,3,550.0,40.0,3,CANCELLED
SAL0003,2026-10-10,C003,Product 4,SAMSUNG,Chennai,4,725.0,60.0,4,COMPLETED
SAL0004,2026-10-11,C004,Product 5,SONY,Hyderabad,5,900.0,0.0,5,RETURNED
SAL0005,2026-10-12,C005,Product 6,LG,Pune,1,1075.0,20.0,1,CANCELLED
SAL0006,2026-10-13,C006,Product 7,SAMSUNG,Mumbai,2,1250.0,40.0,2,COMPLETED
SAL0007,2026-10-14,C007,Product 8,SONY,Kochi,3,1425.0,60.0,3,RETURNED
SAL0008,2026-10-15,C008,Product 1,LG,Delhi,4,200.0,0.0,4,CANCELLED
SAL0009,2026-10-16,C009,Product 2,SAMSUNG,Coimbatore,5,375.0,20.0,5,COMPLETED
SAL0010,2026-10-17,C010,Product 3,SONY,Bengaluru,1,550.0,40.0,1,RETURNED


In [0]:
electronics_df.show(10, False)

+-------+----------+-----------+------------+-------+----------+--------+----------+--------+--------------+-----------+
|sale_id|sale_date |customer_id|product_name|brand  |store_city|quantity|unit_price|discount|warranty_years|sale_status|
+-------+----------+-----------+------------+-------+----------+--------+----------+--------+--------------+-----------+
|SAL0001|2026-10-08|C001       |Product 2   |SONY   |Coimbatore|2       |375.0     |20.0    |2             |RETURNED   |
|SAL0002|2026-10-09|C002       |Product 3   |LG     |Bengaluru |3       |550.0     |40.0    |3             |CANCELLED  |
|SAL0003|2026-10-10|C003       |Product 4   |SAMSUNG|Chennai   |4       |725.0     |60.0    |4             |COMPLETED  |
|SAL0004|2026-10-11|C004       |Product 5   |SONY   |Hyderabad |5       |900.0     |0.0     |5             |RETURNED   |
|SAL0005|2026-10-12|C005       |Product 6   |LG     |Pune      |1       |1075.0    |20.0    |1             |CANCELLED  |
|SAL0006|2026-10-13|C006       |

In [0]:
electronics_df.printSchema()

root
 |-- sale_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- store_city: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- warranty_years: integer (nullable = true)
 |-- sale_status: string (nullable = true)



In [0]:
print(electronics_df.columns)

['sale_id', 'sale_date', 'customer_id', 'product_name', 'brand', 'store_city', 'quantity', 'unit_price', 'discount', 'warranty_years', 'sale_status']


In [0]:
print("Source Count =", electronics_df.count())

Source Count = 40


In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_df = electronics_df \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("batch_id", lit("BATCH_001")) \
    .withColumn("ingestion_timestamp", current_timestamp())

In [0]:
display(electronics_df)

sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status
SAL0001,2026-10-08,C001,Product 2,SONY,Coimbatore,2,375.0,20.0,2,RETURNED
SAL0002,2026-10-09,C002,Product 3,LG,Bengaluru,3,550.0,40.0,3,CANCELLED
SAL0003,2026-10-10,C003,Product 4,SAMSUNG,Chennai,4,725.0,60.0,4,COMPLETED
SAL0004,2026-10-11,C004,Product 5,SONY,Hyderabad,5,900.0,0.0,5,RETURNED
SAL0005,2026-10-12,C005,Product 6,LG,Pune,1,1075.0,20.0,1,CANCELLED
SAL0006,2026-10-13,C006,Product 7,SAMSUNG,Mumbai,2,1250.0,40.0,2,COMPLETED
SAL0007,2026-10-14,C007,Product 8,SONY,Kochi,3,1425.0,60.0,3,RETURNED
SAL0008,2026-10-15,C008,Product 1,LG,Delhi,4,200.0,0.0,4,CANCELLED
SAL0009,2026-10-16,C009,Product 2,SAMSUNG,Coimbatore,5,375.0,20.0,5,COMPLETED
SAL0010,2026-10-17,C010,Product 3,SONY,Bengaluru,1,550.0,40.0,1,RETURNED


In [0]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("electronic_sales.bronze.electronics_sales_bronze")

In [0]:
from pyspark.sql.functions import *

bronze_df = electronics_df \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("batch_id", lit("BATCH_001")) \
    .withColumn("ingestion_timestamp", current_timestamp())



In [0]:
from pyspark.sql.functions import *

duplicate_ids = bronze_df.groupBy("sale_id") \
    .count() \
    .filter("count > 1") \
    .select("sale_id")

validated_df = bronze_df \
    .join(
        duplicate_ids.withColumn("duplicate_flag", lit("Y")),
        ["sale_id"],
        "left"
    ) \
    .withColumn(
        "rejection_reason",
        when(
            col("sale_id").isNull() | (trim(col("sale_id")) == ""),
            "SALE_ID_MISSING"
        ).when(
            col("customer_id").isNull() | (trim(col("customer_id")) == ""),
            "CUSTOMER_ID_MISSING"
        ).when(
            col("product_name").isNull() | (trim(col("product_name")) == ""),
            "PRODUCT_NAME_MISSING"
        ).when(
            col("brand").isNull() | (trim(col("brand")) == ""),
            "BRAND_MISSING"
        ).when(
            col("quantity") <= 0,
            "INVALID_QUANTITY"
        ).when(
            col("unit_price") <= 0,
            "INVALID_UNIT_PRICE"
        ).when(
            col("discount") < 0,
            "INVALID_DISCOUNT"
        ).when(
            col("warranty_years") < 0,
            "INVALID_WARRANTY"
        ).when(
            ~col("sale_status").isin(
                "COMPLETED",
                "RETURNED",
                "CANCELLED"
            ),
            "INVALID_STATUS"
        ).when(
            col("duplicate_flag") == "Y",
            "DUPLICATE_SALE_ID"
        )
    )

In [0]:
valid_df = validated_df.filter(col("rejection_reason").isNull())

rejected_df = validated_df.filter(col("rejection_reason").isNotNull())

In [0]:
valid_df = validated_df.filter(col("rejection_reason").isNull())

valid_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "electronic_sales.bronze.electronics_sales_valid"
    )

In [0]:
rejected_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "electronic_sales.audit.electronics_sales_rejected"
    )

In [0]:
display(
    spark.table(
        "electronic_sales.bronze.electronics_sales_valid"
    )
)

display(
    spark.table(
        "electronic_sales.audit.electronics_sales_rejected"
    )
)

sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason
SAL0001,2026-10-08,C001,Product 2,SONY,Coimbatore,2,375.0,20.0,2,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:41:18.470Z,null,null
SAL0002,2026-10-09,C002,Product 3,LG,Bengaluru,3,550.0,40.0,3,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:41:18.470Z,null,null
SAL0003,2026-10-10,C003,Product 4,SAMSUNG,Chennai,4,725.0,60.0,4,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:41:18.470Z,null,null
SAL0004,2026-10-11,C004,Product 5,SONY,Hyderabad,5,900.0,0.0,5,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:41:18.470Z,null,null
SAL0005,2026-10-12,C005,Product 6,LG,Pune,1,1075.0,20.0,1,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:41:18.470Z,null,null
SAL0006,2026-10-13,C006,Product 7,SAMSUNG,Mumbai,2,1250.0,40.0,2,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:41:18.470Z,null,null
SAL0007,2026-10-14,C007,Product 8,SONY,Kochi,3,1425.0,60.0,3,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:41:18.470Z,null,null
SAL0008,2026-10-15,C008,Product 1,LG,Delhi,4,200.0,0.0,4,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:41:18.470Z,null,null
SAL0009,2026-10-16,C009,Product 2,SAMSUNG,Coimbatore,5,375.0,20.0,5,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:41:18.470Z,null,null
SAL0010,2026-10-17,C010,Product 3,SONY,Bengaluru,1,550.0,40.0,1,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:41:18.470Z,null,null


sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason
null,2026-10-15,C036,Product 5,SAMSUNG,Hyderabad,2,900.0,0.0,2,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:43:37.090Z,null,SALE_ID_MISSING
SAL0037,2026-10-16,C037,null,SONY,Pune,3,1075.0,20.0,3,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:43:37.090Z,null,PRODUCT_NAME_MISSING
SAL0038,2026-10-17,C038,Product 7,LG,Mumbai,-1,1250.0,40.0,4,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:43:37.090Z,null,INVALID_QUANTITY
SAL0039,2026-10-18,C039,Product 8,SAMSUNG,Kochi,5,1425.0,60.0,5,INVALID_VALUE,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:43:37.090Z,Y,INVALID_STATUS
SAL0039,2026-10-19,C040,Product 1,SONY,Delhi,1,200.0,0.0,1,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T05:43:37.090Z,Y,DUPLICATE_SALE_ID


C. Validation Rules

•	sale_id and customer_id are mandatory

•	product_name and brand must not be blank

•	quantity and unit_price must be greater than 0

•	discount and warranty_years must be 0 or greater

•	sale_date must be valid

•	sale_status must be COMPLETED, RETURNED or CANCELLED

•	duplicate sale_id values must be rejected


In [0]:
from pyspark.sql.functions import *

duplicate_ids = bronze_df.groupBy("sale_id") \
                         .count() \
                         .filter("count > 1") \
                         .select("sale_id")

validated_df = bronze_df \
    .join(
        duplicate_ids.withColumn("duplicate_flag", lit("Y")),
        ["sale_id"],
        "left"
    ) \
    .withColumn(
        "rejection_reason",
        when(
            col("sale_id").isNull() | (trim(col("sale_id")) == ""),
            "SALE_ID_MISSING"
        ).when(
            col("customer_id").isNull() | (trim(col("customer_id")) == ""),
            "CUSTOMER_ID_MISSING"
        ).when(
            col("product_name").isNull() | (trim(col("product_name")) == ""),
            "PRODUCT_NAME_MISSING"
        ).when(
            col("brand").isNull() | (trim(col("brand")) == ""),
            "BRAND_MISSING"
        ).when(
            col("quantity") <= 0,
            "INVALID_QUANTITY"
        ).when(
            col("unit_price") <= 0,
            "INVALID_UNIT_PRICE"
        ).when(
            col("discount") < 0,
            "INVALID_DISCOUNT"
        ).when(
            col("warranty_years") < 0,
            "INVALID_WARRANTY"
        ).when(
            to_date(col("sale_date")).isNull(),
            "INVALID_SALE_DATE"
        ).when(
            ~col("sale_status").isin(
                "COMPLETED",
                "RETURNED",
                "CANCELLED"
            ),
            "INVALID_STATUS"
        ).when(
            col("duplicate_flag") == "Y",
            "DUPLICATE_SALE_ID"
        )
    )

In [0]:
valid_df = validated_df.filter(col("rejection_reason").isNull())

rejected_df = validated_df.filter(col("rejection_reason").isNotNull())

D. Reconciliation

•	Show Source Count, Valid Count and Rejected Count.

•	Prove Source Count = Valid Count + Rejected Count.

•	Display PASS or FAIL.

•	Confirm that rejected records do not enter Silver.


In [0]:
source_count = bronze_df.count()

valid_count = valid_df.count()

rejected_count = rejected_df.count()

reconciliation_status = (
    "PASS"
    if source_count == (valid_count + rejected_count)
    else "FAIL"
)

print("Source Count   :", source_count)
print("Valid Count    :", valid_count)
print("Rejected Count :", rejected_count)

print(
    "\nValidation Check :",
    f"{source_count} = {valid_count} + {rejected_count}"
)

print(
    "\nReconciliation Status :",
    reconciliation_status
)

silver_source_count = valid_df.count()

if silver_source_count == valid_count:
    print(
        "\nSilver Validation : PASS - Only valid records entered Silver"
    )
else:
    print(
        "\nSilver Validation : FAIL - Rejected records entered Silver"
    )

Source Count   : 40
Valid Count    : 35
Rejected Count : 5

Validation Check : 40 = 35 + 5

Reconciliation Status : PASS

Silver Validation : PASS - Only valid records entered Silver


In [0]:
source_count = bronze_df.count()
print("Source Count :", source_count)

Source Count : 40


In [0]:
valid_count = valid_df.count()
print("Valid Count :", valid_count)


Valid Count : 35


In [0]:
rejected_count = rejected_df.count()
print("Rejected Count :", rejected_count)

Rejected Count : 5


In [0]:
# Source Count = Valid Count + Rejected Count.

print(
    "\nValidation Check :",
    f"{source_count} = {valid_count} + {rejected_count}"
)



Validation Check : 40 = 35 + 5


In [0]:
# Status

print(
    "\nReconciliation Status :",
    reconciliation_status
)


Reconciliation Status : PASS


In [0]:
# •	Confirm that rejected records do not enter Silver

silver_source_count = valid_df.count()

if silver_source_count == valid_count:
    print(
        "\nSilver Validation : PASS - Only valid records entered Silver"
    )
else:
    print(
        "\nSilver Validation : FAIL - Rejected records entered Silver"
    )


Silver Validation : PASS - Only valid records entered Silver


In [0]:
silver_df = valid_df

E . Silver Requirements

•	Trim product_name; uppercase brand and store_city

•	Cast date and numeric columns

•	Create the calculated column: net_sales = (quantity * unit_price) - discount.

•	Write a Silver Delta table using one row per unique sale_id.


In [0]:
from pyspark.sql.functions import *

silver_df = valid_df \
    .withColumn("product_name", trim(col("product_name"))) \
    .withColumn("brand", upper(col("brand"))) \
    .withColumn("store_city", upper(col("store_city"))) \
    .withColumn("sale_date", to_date(col("sale_date"))) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("unit_price", col("unit_price").cast("double")) \
    .withColumn("discount", col("discount").cast("double")) \
    .withColumn("warranty_years", col("warranty_years").cast("int")) \
    .withColumn(
        "net_sales",
        (col("quantity") * col("unit_price")) - col("discount")
    ) \
    .dropDuplicates(["sale_id"])

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "electronic_sales.silver.electronics_sales"
    )

display(silver_df)

sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales
SAL0003,2026-10-10,C003,Product 4,SAMSUNG,CHENNAI,4,725.0,60.0,4,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:09:56.681Z,null,null,2840.0
SAL0004,2026-10-11,C004,Product 5,SONY,HYDERABAD,5,900.0,0.0,5,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:09:56.681Z,null,null,4500.0
SAL0028,2026-10-07,C028,Product 5,SONY,HYDERABAD,4,900.0,0.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:09:56.681Z,null,null,3600.0
SAL0009,2026-10-16,C009,Product 2,SAMSUNG,COIMBATORE,5,375.0,20.0,5,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:09:56.681Z,null,null,1855.0
SAL0011,2026-10-18,C011,Product 4,LG,CHENNAI,2,725.0,60.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:09:56.681Z,null,null,1390.0
SAL0024,2026-10-03,C024,Product 1,SAMSUNG,DELHI,5,200.0,0.0,5,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:09:56.681Z,null,null,1000.0
SAL0002,2026-10-09,C002,Product 3,LG,BENGALURU,3,550.0,40.0,3,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:09:56.681Z,null,null,1610.0
SAL0008,2026-10-15,C008,Product 1,LG,DELHI,4,200.0,0.0,4,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:09:56.681Z,null,null,800.0
SAL0006,2026-10-13,C006,Product 7,SAMSUNG,MUMBAI,2,1250.0,40.0,2,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:09:56.681Z,null,null,2460.0
SAL0014,2026-10-21,C014,Product 7,LG,MUMBAI,5,1250.0,40.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:09:56.681Z,null,null,6210.0


In [0]:
from pyspark.sql.functions import *

In [0]:
silver_df = valid_df \
    .withColumn("product_name", trim(col("product_name"))) \
    .withColumn("brand", upper(col("brand"))) \
    .withColumn("store_city", upper(col("store_city"))) \
    .withColumn("sale_date", to_date(col("sale_date"))) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("unit_price", col("unit_price").cast("double")) \
    .withColumn("discount", col("discount").cast("double")) \
    .withColumn("warranty_years", col("warranty_years").cast("int")) \
    .withColumn(
        "net_sales",
        (col("quantity") * col("unit_price")) - col("discount")
    ) \
    .dropDuplicates(["sale_id"])

In [0]:
# Silver Delta table using one row per unique sale_id.


silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "electronic_sales.silver.electronics_sales"
    )

display(silver_df)

sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales
SAL0003,2026-10-10,C003,Product 4,SAMSUNG,CHENNAI,4,725.0,60.0,4,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:37.463Z,null,null,2840.0
SAL0004,2026-10-11,C004,Product 5,SONY,HYDERABAD,5,900.0,0.0,5,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:37.463Z,null,null,4500.0
SAL0028,2026-10-07,C028,Product 5,SONY,HYDERABAD,4,900.0,0.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:37.463Z,null,null,3600.0
SAL0009,2026-10-16,C009,Product 2,SAMSUNG,COIMBATORE,5,375.0,20.0,5,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:37.463Z,null,null,1855.0
SAL0011,2026-10-18,C011,Product 4,LG,CHENNAI,2,725.0,60.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:37.463Z,null,null,1390.0
SAL0024,2026-10-03,C024,Product 1,SAMSUNG,DELHI,5,200.0,0.0,5,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:37.463Z,null,null,1000.0
SAL0002,2026-10-09,C002,Product 3,LG,BENGALURU,3,550.0,40.0,3,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:37.463Z,null,null,1610.0
SAL0008,2026-10-15,C008,Product 1,LG,DELHI,4,200.0,0.0,4,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:37.463Z,null,null,800.0
SAL0006,2026-10-13,C006,Product 7,SAMSUNG,MUMBAI,2,1250.0,40.0,2,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:37.463Z,null,null,2460.0
SAL0014,2026-10-21,C014,Product 7,LG,MUMBAI,5,1250.0,40.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:37.463Z,null,null,6210.0


F. Gold Requirement

•	Create Gold output: electronics_revenue_by_brand.

•	Gold grain: one row per brand.

•	Required measures: total_sales_count, total_quantity, total_revenue, avg_sale_value.

•	Which brand generated the highest total revenue?


In [0]:
from pyspark.sql.functions import *

gold_df = silver_df.groupBy("brand") \
    .agg(
        count("sale_id").alias("total_sales_count"),
        sum("quantity").alias("total_quantity"),
        round(sum("net_sales"), 2).alias("total_revenue"),
        round(avg("net_sales"), 2).alias("avg_sale_value")
    )

In [0]:
# •	Create Gold output: electronics_revenue_by_brand.

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "electronic_sales.gold.electronics_revenue_by_brand"
    )
display(gold_df)

brand,total_sales_count,total_quantity,total_revenue,avg_sale_value
SAMSUNG,11,34,21315.0,1937.73
SONY,12,37,31365.0,2613.75
LG,12,34,26390.0,2199.17


In [0]:
gold_df.select(
    "brand",
    "total_sales_count"
).show(truncate=False)

+-------+-----------------+
|brand  |total_sales_count|
+-------+-----------------+
|SAMSUNG|11               |
|SONY   |12               |
|LG     |12               |
+-------+-----------------+



In [0]:
gold_df.select(
    "brand",
    "total_quantity"
).show(truncate=False)

+-------+--------------+
|brand  |total_quantity|
+-------+--------------+
|SAMSUNG|34            |
|SONY   |37            |
|LG     |34            |
+-------+--------------+



In [0]:
gold_df.select(
    "brand",
    "total_revenue"
).show(truncate=False)

+-------+-------------+
|brand  |total_revenue|
+-------+-------------+
|SAMSUNG|21315.0      |
|SONY   |31365.0      |
|LG     |26390.0      |
+-------+-------------+



In [0]:
gold_df.select(
    "brand",
    "avg_sale_value"
).show(truncate=False)

+-------+--------------+
|brand  |avg_sale_value|
+-------+--------------+
|SAMSUNG|1937.73       |
|SONY   |2613.75       |
|LG     |2199.17       |
+-------+--------------+



In [0]:
gold_df.orderBy(
    desc("total_revenue")
).show(1, truncate=False)

+-----+-----------------+--------------+-------------+--------------+
|brand|total_sales_count|total_quantity|total_revenue|avg_sale_value|
+-----+-----------------+--------------+-------------+--------------+
|SONY |12               |37            |31365.0      |2613.75       |
+-----+-----------------+--------------+-------------+--------------+
only showing top 1 row


G. Incremental Load and MERGE

•	Read and inspect electronics_sales_incremental.csv.

•	Add BATCH_002 audit metadata and apply the same validation/cleaning logic.

•	Align the incoming source with the existing Silver target schema.

•	Identify existing and new sale_id values.

•	Use sale_id as the Delta MERGE condition.

•	WHEN MATCHED: update changed values such as quantity, price, discount, warranty_years or status.

•	WHEN NOT MATCHED: insert the new record.

•	Refresh the Gold output from the updated Silver table.

 /Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv


In [0]:
# Read and inspect electronics_sales_incremental.csv


incremental_df = spark.read.csv(
    "/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv",
    header=True,
    inferSchema=True
)

display(incremental_df)

sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status
SAL0001,2026-10-08,C001,Product 2,SONY,Coimbatore,4,375.0,20.0,2,CANCELLED
SAL0002,2026-10-09,C002,Product 3,LG,Bengaluru,5,550.0,40.0,3,COMPLETED
SAL0003,2026-10-10,C003,Product 4,SAMSUNG,Chennai,6,725.0,60.0,4,RETURNED
SAL0004,2026-10-11,C004,Product 5,SONY,Hyderabad,7,900.0,0.0,5,CANCELLED
SAL0101,2026-10-24,C001,Product 6,LG,Pune,2,1075.0,20.0,2,CANCELLED
SAL0102,2026-10-25,C002,Product 7,SAMSUNG,Mumbai,3,1250.0,40.0,3,COMPLETED
SAL0103,2026-10-26,C003,Product 8,SONY,Kochi,4,1425.0,60.0,4,RETURNED
SAL0104,2026-10-27,C004,Product 1,LG,Delhi,5,200.0,0.0,5,CANCELLED
SAL0105,2026-10-28,C005,Product 2,SAMSUNG,Coimbatore,1,375.0,20.0,1,COMPLETED
SAL0106,2026-10-01,C006,Product 3,SONY,Bengaluru,2,550.0,40.0,2,RETURNED


Incremental Source Count = 12
root
 |-- sale_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- store_city: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- warranty_years: integer (nullable = true)
 |-- sale_status: string (nullable = true)

['sale_id', 'sale_date', 'customer_id', 'product_name', 'brand', 'store_city', 'quantity', 'unit_price', 'discount', 'warranty_years', 'sale_status']


In [0]:
print("Incremental Source Count =", incremental_df.count())

incremental_df.printSchema()

Incremental Source Count = 12
root
 |-- sale_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- store_city: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- warranty_years: integer (nullable = true)
 |-- sale_status: string (nullable = true)



In [0]:
print(incremental_df.columns)

['sale_id', 'sale_date', 'customer_id', 'product_name', 'brand', 'store_city', 'quantity', 'unit_price', 'discount', 'warranty_years', 'sale_status']


In [0]:
# Add BATCH_002 audit metadata and apply the same validation/cleaning logic.


from pyspark.sql.functions import *

In [0]:
duplicate_ids = incremental_df.groupBy("sale_id") \
    .count() \
    .filter("count > 1") \
    .select("sale_id")

In [0]:
valid_incremental_df = (
    incremental_df
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("batch_id", lit("BATCH_002"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .join(
        duplicate_ids.withColumn("duplicate_flag", lit("Y")),
        ["sale_id"],
        "left"
    )
    .withColumn(
        "rejection_reason",
        when(col("sale_id").isNull() | (trim(col("sale_id")) == ""), "SALE_ID_MISSING")
        .when(col("customer_id").isNull() | (trim(col("customer_id")) == ""), "CUSTOMER_ID_MISSING")
        .when(col("product_name").isNull() | (trim(col("product_name")) == ""), "PRODUCT_NAME_MISSING")
        .when(col("brand").isNull() | (trim(col("brand")) == ""), "BRAND_MISSING")
        .when(col("quantity") <= 0, "INVALID_QUANTITY")
        .when(col("unit_price") <= 0, "INVALID_UNIT_PRICE")
        .when(col("discount") < 0, "INVALID_DISCOUNT")
        .when(col("warranty_years") < 0, "INVALID_WARRANTY")
        .when(to_date(col("sale_date")).isNull(), "INVALID_SALE_DATE")
        .when(~col("sale_status").isin("COMPLETED","RETURNED","CANCELLED"), "INVALID_STATUS")
        .when(col("duplicate_flag") == "Y", "DUPLICATE_SALE_ID")
    )
    .filter(col("rejection_reason").isNull())
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("brand", upper(col("brand")))
    .withColumn("store_city", upper(col("store_city")))
    .withColumn("sale_date", to_date(col("sale_date")))
    .withColumn(
        "net_sales",
        (col("quantity") * col("unit_price")) - col("discount")
    )
)

display(valid_incremental_df)

sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales
SAL0001,2026-10-08,C001,Product 2,SONY,COIMBATORE,4,375.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T06:53:37.256Z,null,null,1480.0
SAL0002,2026-10-09,C002,Product 3,LG,BENGALURU,5,550.0,40.0,3,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T06:53:37.256Z,null,null,2710.0
SAL0003,2026-10-10,C003,Product 4,SAMSUNG,CHENNAI,6,725.0,60.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T06:53:37.256Z,null,null,4290.0
SAL0004,2026-10-11,C004,Product 5,SONY,HYDERABAD,7,900.0,0.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T06:53:37.256Z,null,null,6300.0
SAL0101,2026-10-24,C001,Product 6,LG,PUNE,2,1075.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T06:53:37.256Z,null,null,2130.0
SAL0102,2026-10-25,C002,Product 7,SAMSUNG,MUMBAI,3,1250.0,40.0,3,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T06:53:37.256Z,null,null,3710.0
SAL0103,2026-10-26,C003,Product 8,SONY,KOCHI,4,1425.0,60.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T06:53:37.256Z,null,null,5640.0
SAL0104,2026-10-27,C004,Product 1,LG,DELHI,5,200.0,0.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T06:53:37.256Z,null,null,1000.0
SAL0105,2026-10-28,C005,Product 2,SAMSUNG,COIMBATORE,1,375.0,20.0,1,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T06:53:37.256Z,null,null,355.0
SAL0106,2026-10-01,C006,Product 3,SONY,BENGALURU,2,550.0,40.0,2,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T06:53:37.256Z,null,null,1060.0


In [0]:
# Align the incoming source with the existing Silver target schema.

silver_df = spark.table(
    "electronic_sales.silver.electronics_sales"
)

valid_incremental_df = valid_incremental_df.select(
    silver_df.columns
)

display(valid_incremental_df)

sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales
SAL0001,2026-10-08,C001,Product 2,SONY,COIMBATORE,4,375.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:11.469Z,null,null,1480.0
SAL0002,2026-10-09,C002,Product 3,LG,BENGALURU,5,550.0,40.0,3,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:11.469Z,null,null,2710.0
SAL0003,2026-10-10,C003,Product 4,SAMSUNG,CHENNAI,6,725.0,60.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:11.469Z,null,null,4290.0
SAL0004,2026-10-11,C004,Product 5,SONY,HYDERABAD,7,900.0,0.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:11.469Z,null,null,6300.0
SAL0101,2026-10-24,C001,Product 6,LG,PUNE,2,1075.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:11.469Z,null,null,2130.0
SAL0102,2026-10-25,C002,Product 7,SAMSUNG,MUMBAI,3,1250.0,40.0,3,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:11.469Z,null,null,3710.0
SAL0103,2026-10-26,C003,Product 8,SONY,KOCHI,4,1425.0,60.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:11.469Z,null,null,5640.0
SAL0104,2026-10-27,C004,Product 1,LG,DELHI,5,200.0,0.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:11.469Z,null,null,1000.0
SAL0105,2026-10-28,C005,Product 2,SAMSUNG,COIMBATORE,1,375.0,20.0,1,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:11.469Z,null,null,355.0
SAL0106,2026-10-01,C006,Product 3,SONY,BENGALURU,2,550.0,40.0,2,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:11.469Z,null,null,1060.0


In [0]:
# •	Identify existing and new sale_id values.

update_candidates = valid_incremental_df.alias("src") \
    .join(
        silver_df.alias("tgt"),
        "sale_id",
        "inner"
    )

insert_candidates = valid_incremental_df.alias("src") \
    .join(
        silver_df.alias("tgt"),
        "sale_id",
        "left_anti"
    )

print("Update Candidate Count :", update_candidates.count())
print("Insert Candidate Count :", insert_candidates.count())

display(update_candidates)
display(insert_candidates)

Update Candidate Count : 4
Insert Candidate Count : 8


sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales
SAL0001,2026-10-08,C001,Product 2,SONY,COIMBATORE,4,375.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:19.380Z,null,null,1480.0,2026-10-08,C001,Product 2,SONY,COIMBATORE,2,375.0,20.0,2,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:36.075Z,null,null,730.0
SAL0002,2026-10-09,C002,Product 3,LG,BENGALURU,5,550.0,40.0,3,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:19.380Z,null,null,2710.0,2026-10-09,C002,Product 3,LG,BENGALURU,3,550.0,40.0,3,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:36.075Z,null,null,1610.0
SAL0003,2026-10-10,C003,Product 4,SAMSUNG,CHENNAI,6,725.0,60.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:19.380Z,null,null,4290.0,2026-10-10,C003,Product 4,SAMSUNG,CHENNAI,4,725.0,60.0,4,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:36.075Z,null,null,2840.0
SAL0004,2026-10-11,C004,Product 5,SONY,HYDERABAD,7,900.0,0.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:19.380Z,null,null,6300.0,2026-10-11,C004,Product 5,SONY,HYDERABAD,5,900.0,0.0,5,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_initial.csv,BATCH_001,2026-11-23T06:10:36.075Z,null,null,4500.0


sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales
SAL0101,2026-10-24,C001,Product 6,LG,PUNE,2,1075.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:20.380Z,null,null,2130.0
SAL0102,2026-10-25,C002,Product 7,SAMSUNG,MUMBAI,3,1250.0,40.0,3,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:20.380Z,null,null,3710.0
SAL0103,2026-10-26,C003,Product 8,SONY,KOCHI,4,1425.0,60.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:20.380Z,null,null,5640.0
SAL0104,2026-10-27,C004,Product 1,LG,DELHI,5,200.0,0.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:20.380Z,null,null,1000.0
SAL0105,2026-10-28,C005,Product 2,SAMSUNG,COIMBATORE,1,375.0,20.0,1,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:20.380Z,null,null,355.0
SAL0106,2026-10-01,C006,Product 3,SONY,BENGALURU,2,550.0,40.0,2,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:20.380Z,null,null,1060.0
SAL0107,2026-10-02,C007,Product 4,LG,CHENNAI,3,725.0,60.0,3,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:20.380Z,null,null,2115.0
SAL0108,2026-10-03,C008,Product 5,SAMSUNG,HYDERABAD,4,900.0,0.0,4,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:20.380Z,null,null,3600.0


In [0]:
display(update_candidates)

sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales
SAL0001,2026-10-08,C001,Product 2,SONY,COIMBATORE,4,375.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:08:57.783Z,null,null,1480.0,2026-10-08,C001,Product 2,SONY,COIMBATORE,4,375.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:35.250Z,null,null,1480.0
SAL0002,2026-10-09,C002,Product 3,LG,BENGALURU,5,550.0,40.0,3,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:08:57.783Z,null,null,2710.0,2026-10-09,C002,Product 3,LG,BENGALURU,5,550.0,40.0,3,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:35.250Z,null,null,2710.0
SAL0003,2026-10-10,C003,Product 4,SAMSUNG,CHENNAI,6,725.0,60.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:08:57.783Z,null,null,4290.0,2026-10-10,C003,Product 4,SAMSUNG,CHENNAI,6,725.0,60.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:35.250Z,null,null,4290.0
SAL0004,2026-10-11,C004,Product 5,SONY,HYDERABAD,7,900.0,0.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:08:57.783Z,null,null,6300.0,2026-10-11,C004,Product 5,SONY,HYDERABAD,7,900.0,0.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:35.250Z,null,null,6300.0
SAL0101,2026-10-24,C001,Product 6,LG,PUNE,2,1075.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:08:57.783Z,null,null,2130.0,2026-10-24,C001,Product 6,LG,PUNE,2,1075.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:35.250Z,null,null,2130.0
SAL0102,2026-10-25,C002,Product 7,SAMSUNG,MUMBAI,3,1250.0,40.0,3,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:08:57.783Z,null,null,3710.0,2026-10-25,C002,Product 7,SAMSUNG,MUMBAI,3,1250.0,40.0,3,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:35.250Z,null,null,3710.0
SAL0103,2026-10-26,C003,Product 8,SONY,KOCHI,4,1425.0,60.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:08:57.783Z,null,null,5640.0,2026-10-26,C003,Product 8,SONY,KOCHI,4,1425.0,60.0,4,RETURNED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:35.250Z,null,null,5640.0
SAL0104,2026-10-27,C004,Product 1,LG,DELHI,5,200.0,0.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:08:57.783Z,null,null,1000.0,2026-10-27,C004,Product 1,LG,DELHI,5,200.0,0.0,5,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:35.250Z,null,null,1000.0
SAL0105,2026-10-28,C005,Product 2,SAMSUNG,COIMBATORE,1,375.0,20.0,1,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:08:57.783Z,null,null,355.0,2026-10-28,C005,Product 2,SAMSUNG,COIMBATORE,1,375.0,20.0,1,COMPLETED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:35.250Z,nu

In [0]:
# •	Use sale_id as the Delta MERGE condition.

from delta.tables import DeltaTable

silver_delta = DeltaTable.forName(
    spark,
    "electronic_sales.silver.electronics_sales"
)

(
    silver_delta.alias("target")
    .merge(
        valid_incremental_df.alias("source"),
        "target.sale_id = source.sale_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# •	Refresh the Gold output from the updated Silver table.

updated_silver_df = spark.table(
    "electronic_sales.silver.electronics_sales"
)

gold_df = updated_silver_df.groupBy("brand") \
    .agg(
        count("sale_id").alias("total_sales_count"),
        sum("quantity").alias("total_quantity"),
        round(sum("net_sales"), 2).alias("total_revenue"),
        round(avg("net_sales"), 2).alias("avg_sale_value")
    )

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "electronic_sales.gold.electronics_revenue_by_brand"
    )

In [0]:
# highest revenue after inserting the values

gold_df.orderBy(
    desc("total_revenue")
).show(1, False)

+-----+-----------------+--------------+-------------+--------------+
|brand|total_sales_count|total_quantity|total_revenue|avg_sale_value|
+-----+-----------------+--------------+-------------+--------------+
|SONY |14               |47            |40615.0      |2901.07       |
+-----+-----------------+--------------+-------------+--------------+
only showing top 1 row


H. Required Final Evidence

•	Initial source, valid and rejected counts with reconciliation result.

•	Rejected rows with rejection_reason.

•	Silver count before MERGE.

•	Incremental source count, update candidate count and insert candidate count.

•	Silver count after MERGE and the expected-count calculation.

•	One updated sale and one newly inserted sale.

•	Confirmation that no duplicate sale_id remains in Silver.

•	Refreshed Gold output and the answer to the business question.

•	A short verbal explanation of Bronze, Silver, Gold, reconciliation and MERGE logic.


In [0]:
# • Initial source, valid and rejected counts with reconciliation result.

print("Source Count   :", source_count)
print("Valid Count    :", valid_count)
print("Rejected Count :", rejected_count)

print(
    "Reconciliation Result :",
    "PASS" if source_count == (valid_count + rejected_count)
    else "FAIL"
)

Source Count   : 40
Valid Count    : 35
Rejected Count : 5
Reconciliation Result : PASS


In [0]:
# Rejected rows with rejection_reason.


display(
    rejected_df.select(
        "sale_id",
        "customer_id",
        "product_name",
        "rejection_reason"
    )
)

sale_id,customer_id,product_name,rejection_reason
null,C036,Product 5,SALE_ID_MISSING
SAL0037,C037,null,PRODUCT_NAME_MISSING
SAL0038,C038,Product 7,INVALID_QUANTITY
SAL0039,C039,Product 8,INVALID_STATUS
SAL0039,C040,Product 1,DUPLICATE_SALE_ID


In [0]:
# Silver count before MERGE.
silver_before_count = spark.table(
    "electronic_sales.silver.electronics_sales"
).count()

print(
    "Silver Count Before MERGE :",
    silver_before_count
)

Silver Count Before MERGE : 43


In [0]:
#  Incremental source count, update candidate count and insert candidate count.
# Candidate counts against the Silver population as it was before the MERGE

before_merge_ids = spark.table(
    "electronic_sales.bronze.electronics_sales_valid"
).select("sale_id")

incremental_ids = incremental_df.select("sale_id")

update_count = incremental_ids.join(
    before_merge_ids,
    "sale_id",
    "inner"
).count()

insert_count = incremental_ids.join(
    before_merge_ids,
    "sale_id",
    "left_anti"
).count()

print(
    "Incremental Source Count :",
    incremental_df.count()
)

print(
    "Update Candidate Count :",
    update_count
)

print(
    "Insert Candidate Count :",
    insert_count
)

Incremental Source Count : 12
Update Candidate Count : 4
Insert Candidate Count : 8


In [0]:
# Silver count after MERGE and the expected-count calculation.

silver_after_count = spark.table(
    "electronic_sales.silver.electronics_sales"
).count()

expected_count = (
    silver_before_count +
    insert_count
)

print(
    "Expected Silver Count :",
    expected_count
)

print(
    "Actual Silver Count :",
    silver_after_count
)

print(
    "Count Validation :",
    "PASS"
    if expected_count == silver_after_count
    else "FAIL"
)

Expected Silver Count : 51
Actual Silver Count : 43
Count Validation : FAIL


In [0]:
#  •	One updated sale and one newly inserted sale.
 

display(
    update_candidates.limit(1)
)

sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales
SAL0001,2026-10-08,C001,Product 2,SONY,COIMBATORE,4,375.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:36:14.725Z,null,null,1480.0,2026-10-08,C001,Product 2,SONY,COIMBATORE,4,375.0,20.0,2,CANCELLED,dbfs:/Volumes/electronic_sales/bronze/raw_data/electronics_sales_incremental.csv,BATCH_002,2026-11-23T07:07:35.250Z,null,null,1480.0


In [0]:
 

display(
    insert_candidates.limit(1)
)

sale_id,sale_date,customer_id,product_name,brand,store_city,quantity,unit_price,discount,warranty_years,sale_status,source_file,batch_id,ingestion_timestamp,duplicate_flag,rejection_reason,net_sales


In [0]:
# •	Confirmation that no duplicate sale_id remains in Silver.

duplicate_count = spark.table(
    "electronic_sales.silver.electronics_sales"
).groupBy(
    "sale_id"
).count().filter(
    "count > 1"
).count()

print(
    "Duplicate Sale_ID Count :",
    duplicate_count
)

print(
    "Duplicate Validation :",
    "PASS"
    if duplicate_count == 0
    else "FAIL"
)


Duplicate Sale_ID Count : 0
Duplicate Validation : PASS


In [0]:
# •	Refreshed Gold output and the answer to the business question.

gold_df = spark.table(
    "electronic_sales.gold.electronics_revenue_by_brand"
)

display(
    gold_df.orderBy(
        desc("total_revenue")
    )
)

brand,total_sales_count,total_quantity,total_revenue,avg_sale_value
SONY,14,47,40615.0,2901.07
LG,15,46,32735.0,2182.33
SAMSUNG,14,44,30430.0,2173.57


In [0]:
gold_df.orderBy(
    desc("total_revenue")
).show(1, False)

+-----+-----------------+--------------+-------------+--------------+
|brand|total_sales_count|total_quantity|total_revenue|avg_sale_value|
+-----+-----------------+--------------+-------------+--------------+
|SONY |14               |47            |40615.0      |2901.07       |
+-----+-----------------+--------------+-------------+--------------+
only showing top 1 row


**	A short verbal explanation of Bronze, Silver, Gold, reconciliation and MERGE logic.**

**Bronze Layer**

Bronze stores the raw source data exactly as received from the CSV file. It includes audit columns such as source_file, batch_id, and ingestion_timestamp to provide data lineage and traceability.

**Validation & Reconciliation**

Validation rules are applied to identify invalid records. Rejected records are stored separately with a rejection_reason. Reconciliation verifies that Source Count equals Valid Count plus Rejected Count, ensuring no records are lost during processing.

**Silver Layer**

Silver contains cleansed and standardized data. Product names are trimmed, brand and store city values are converted to uppercase, datatypes are corrected, and the calculated column net_sales = (quantity * unit_price) - discount is created. Only one record per sale_id is retained.

**Gold Layer**

Gold contains business-ready aggregated data. In this use case, revenue metrics are generated at the brand level, including total sales count, total quantity, total revenue, and average sale value.

**MERGE Logic**

During incremental processing, existing sale_id values are identified as update candidates and new sale_id values as insert candidates. Delta MERGE uses sale_id as the business key. Existing records are updated, and new records are inserted. After MERGE, the Gold table is refreshed to reflect the latest business results.